# Demo 01 : A research agent

We use local manuals and service tickets to answer a question that spans both sources. Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


## a · research-agent

We keep the corpus small and search only local files. The graph separates planning, research and report writing, following the shape of a deep research agent without web access.

In [2]:
import csv
import operator
import re
from pathlib import Path
from typing import Annotated

from langchain.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel
from typing_extensions import TypedDict

manuals_dir = Path(data_folder("demo-world/manuals"))
manuals = [(p.name, p.read_text(encoding="utf-8")) for p in sorted(manuals_dir.glob("*.md"))]
with open(data("demo-world/tickets.csv"), encoding="utf-8") as f:
    tickets = list(csv.DictReader(f))

CODE = re.compile(r"F-[A-Z]{2}-\d{3}")
STOP = {"what", "which", "does", "that", "this", "with", "from", "most", "local", "apply", "applies"}

def search(query, limit=2):
    codes = [c.lower() for c in CODE.findall(query)]
    words = {w for w in re.findall(r"[a-z0-9-]+", query.lower()) if len(w) > 3 and w not in STOP}
    def score(row):
        body = row[1].lower()
        return (sum(c in body for c in codes), sum(body.count(w) for w in words))
    return sorted(manuals, key=score, reverse=True)[:limit]

QUESTION = "Which repeated fault took the most recorded mechanic hours, and what local procedure applies?"
print("manual files:", len(manuals), "ticket rows:", len(tickets))

manual files: 31 ticket rows: 43


A planner splits the question into a ticket question and a manual question. The manual question depends on the ticket answer : we cannot look up a procedure before we know which fault leads. So the two research steps run in order, and the manual step searches with the planner's question plus the fault code the ticket step found.

In [3]:
class Plan(BaseModel):
    ticket_question: str
    manual_question: str

class ResearchState(TypedDict):
    question: str
    plan: dict
    top_code: str
    query: str
    notes: Annotated[list[dict], operator.add]
    report: str

def plan_node(state):
    plan = llm.with_structured_output(Plan).invoke(
        "Split the goal into one short question for the service tickets and one for the repair manuals. "
        "The goal is to answer: " + state["question"])
    return {"plan": plan.model_dump()}

def research(question, files, context):
    return llm.invoke([
        SystemMessage("Use only the supplied context. State what it supports and name the source file."),
        HumanMessage(f"Question: {question}\nFiles: {files}\nContext: {context}")
    ]).content

def tickets_node(state):
    hours = {}
    for row in tickets:
        key = (row["error_code"], row["symptom"])
        hours[key] = hours.get(key, 0) + float(row["hours"])
    facts = sorted(hours.items(), key=lambda item: item[1], reverse=True)[:4]
    finding = research(state["plan"]["ticket_question"], ["tickets.csv"], str(facts))
    return {"top_code": facts[0][0][0], "notes": [{"source": "tickets", "files": ["tickets.csv"], "finding": finding}]}

def manuals_node(state):
    query = f"{state['plan']['manual_question']} {state['top_code']}"
    hits = search(query)
    files = [name for name, _ in hits]
    context = "\n\n".join(f"[{name}]\n{body[:1000]}" for name, body in hits)
    finding = research(state["plan"]["manual_question"] + f" The leading fault is {state['top_code']}.", files, context)
    return {"query": query, "notes": [{"source": "manuals", "files": files, "finding": finding}]}

def write_report(state):
    notes = "\n".join(f"{n['finding']} [" + ", ".join(n["files"]) + "]" for n in state["notes"])
    report = llm.invoke("Answer the question using only these notes. Keep file citations beside claims. "
                        f"Question: {state['question']}\nNotes:\n{notes}").content
    return {"report": report}

graph = StateGraph(ResearchState)
graph.add_node("plan", plan_node)
graph.add_node("tickets", tickets_node)
graph.add_node("manuals", manuals_node)
graph.add_node("write_report", write_report)
graph.add_edge(START, "plan")
graph.add_edge("plan", "tickets")
graph.add_edge("tickets", "manuals")
graph.add_edge("manuals", "write_report")
graph.add_edge("write_report", END)
result = graph.compile().invoke({"question": QUESTION, "plan": {}, "top_code": "", "query": "", "notes": [], "report": ""})
print("plan:", result["plan"])
print("top fault:", result["top_code"])
print("manual query:", result["query"])
print("report:\n", result["report"])

plan: {'ticket_question': 'Which repeated fault took the most recorded mechanic hours?', 'manual_question': 'What local procedure applies to that fault?'}
top fault: F-RR-120
manual query: What local procedure applies to that fault? F-RR-120
report:
 **F-RR-120 — “Door controller not responding”** took the most recorded mechanic time: **27.5 hours**. [tickets.csv]

The local procedure is **PROC-RR-BP-03**, for replacing the RR-14 auxiliary battery pack. It directs you to charge or replace the pack. For a replacement, book a workshop bay, isolate the bus, fit the new pack, run the self-test, and record the part and fleet numbers. Allow **2 hours**. [rr-14-procedure-rr-bp-03.md]


We check provenance in code. The retrieved files must belong to the model whose fault leads, and the report should cite one of them.

In [4]:
sources = sorted({name for note in result["notes"] for name in note["files"]})
retrieved = result["notes"][-1]["files"]
assert any(name.startswith("rr-14") for name in retrieved), retrieved
print("sources:", sources)
print("report cites an rr-14 file:", any(name in result["report"] for name in retrieved))

sources: ['rr-14-error-codes.md', 'rr-14-procedure-rr-bp-03.md', 'tickets.csv']
report cites an rr-14 file: True


A source list checks provenance, not whether each source supports the sentence beside it. We read the cited passage before using the report as an operational recommendation.

## Things to try

- Search the manuals with the planner's question alone, without the fault code, and compare the files
- Change retrieval to return one manual and compare the evidence